# Supply Chain Analysis


In [327]:
import pandas as pd 
import numpy as np 

In [328]:
df = pd.read_csv('messy_supply_chain_data.csv')

In [432]:
df.head()



,order_id,order_date,ship_date,delivery_date,supplier_name,supplier_location,warehouse_location,product_category,product_id,quantity,...,payment_method,customer_type,region,returned,supplier_city,supplier_country,returned_flag,payment_group,customer_segment,is_b2b
0,ORD-19694,2023-05-27,2023-05-30,2023-06-07,Industrial Supplies Ltd,"New York, USA",WH-ALPHA,Electronics,PROD-913,94.0,...,UPI,Corporate,North,Yes,New York,USA,1,Online,B2B,1
1,ORD-28320,2023-09-20,2023-09-22,2023-10-08,Global Tech,"Shanghai, CN",WH-BETA,Electronics,PROD-427,54.0,...,COD,Wholesale,West,No,Shanghai,CN,0,Offline,B2B,1
2,ORD-13847,2022-12-17,2022-12-19,2022-12-28,Industrial Supplies Ltd Inc,"Berlin, Germany",MAIN_HUB,Clothing,PROD-939,95.0,...,UPI,Retail,West,No,Berlin,Germany,0,Online,B2C,0
3,ORD-29360,2022-08-24,2022-08-26,2022-09-10,Global Tech,"Shanghai, CN",WH-ALPHA,Industrial Tools,PROD-649,95.0,...,Card,Wholesale,North,Yes,Shanghai,CN,1,Online,B2B,1
4,ORD-74722,2023-03-23,2023-03-24,2023-04-12,Industrial Supplies Ltd Inc,"Paris, France",MAIN_HUB,Electronics,PROD-938,28.0,...,Card,Corporate,North,No,Paris,France,0,Online,B2B,1


In [330]:
print("Total Number of datapoints :",df.shape[0])
print("Total Number of features :",df.shape[1])

Total Number of datapoints : 4500
Total Number of features : 20


In [438]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4178 entries, 0 to 4499
Data columns (total 26 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   order_id            4178 non-null   object        
 1   order_date          4178 non-null   datetime64[ns]
 2   ship_date           4178 non-null   datetime64[ns]
 3   delivery_date       4178 non-null   datetime64[ns]
 4   supplier_name       4178 non-null   object        
 5   supplier_location   4178 non-null   object        
 6   warehouse_location  4178 non-null   object        
 7   product_category    4178 non-null   object        
 8   product_id          4178 non-null   object        
 9   quantity            4178 non-null   float64       
 10  unit_price          4178 non-null   float64       
 11  total_cost          4178 non-null   float64       
 12  shipping_cost       4178 non-null   float64       
 13  shipping_mode       4178 non-null   object        
 1

In [439]:
df.isnull().sum()

order_id              0
order_date            0
ship_date             0
delivery_date         0
supplier_name         0
supplier_location     0
warehouse_location    0
product_category      0
product_id            0
quantity              0
unit_price            0
total_cost            0
shipping_cost         0
shipping_mode         0
delivery_status       0
delay_days            0
payment_method        0
customer_type         0
region                0
returned              0
supplier_city         0
supplier_country      0
returned_flag         0
payment_group         0
customer_segment      0
is_b2b                0
dtype: int64

In [333]:
df.columns = df.columns.str.strip().str.lower()
df.columns

Index(['order_id', 'order_date', 'ship_date', 'delivery_date', 'supplier_name',
       'supplier_location', 'warehouse_location', 'product_category',
       'product_id', 'quantity', 'unit_price', 'total_cost', 'shipping_cost',
       'shipping_mode', 'delivery_status', 'delay_days', 'payment_method',
       'customer_type', 'region', 'returned'],
      dtype='object')

In [334]:
df['order_id'].duplicated().sum()

np.int64(329)

In [335]:
df = df.drop_duplicates()

In [336]:
df['order_id'] = df['order_id'].str.strip().str.upper()

In [337]:
df['order_id'].isnull().sum()

np.int64(0)

In [338]:
df['order_date'].head(10)

0    2023/05/27
1    2023/09/20
2           NaN
3    08/24/2022
4    23-03-2023
5           NaN
6    04/06/2022
7    07/31/2023
8    03-03-2023
9    19-10-2022
Name: order_date, dtype: object

In [339]:
df['order_date'].value_counts()

order_date
01/11/2022    8
08/22/2022    8
2022/05/21    6
28-07-2023    6
07/30/2023    6
             ..
07/14/2022    1
2022/04/01    1
2023/11/15    1
29-11-2023    1
2022/04/27    1
Name: count, Length: 1681, dtype: int64

In [340]:
def parse_date(x):
    try:
        x = str(x).strip()
        
        if x in ['nan', 'None', '']:
            return pd.NaT
        
        # Case 1: YYYY-MM-DD
        if '-' in x:
            parts = x.split('-')
            
            # If first part is 4 digits → YYYY-MM-DD
            if len(parts[0]) == 4:
                return pd.to_datetime(x, format='%Y-%m-%d', errors='coerce')
            
            # Else → DD-MM-YYYY
            else:
                return pd.to_datetime(x, format='%d-%m-%Y', errors='coerce')
        
        # Case 2: Slash formats
        elif '/' in x:
            parts = x.split('/')
            
            if len(parts[0]) == 4:
                return pd.to_datetime(x, format='%Y/%m/%d', errors='coerce')
            else:
                return pd.to_datetime(x, format='%m/%d/%Y', errors='coerce')
        
        else:
            return pd.NaT
    
    except:
        return pd.NaT

In [341]:
df['order_date'] = df['order_date'].apply(parse_date)


In [342]:
df['order_date'].head(10)

0   2023-05-27
1   2023-09-20
2          NaT
3   2022-08-24
4   2023-03-23
5          NaT
6   2022-04-06
7   2023-07-31
8   2023-03-03
9   2022-10-19
Name: order_date, dtype: datetime64[ns]

In [343]:
df['order_date'].isnull().sum()

np.int64(1189)

In [344]:
df['ship_date'] = df['ship_date'].apply(parse_date)

In [345]:
df['ship_date'].dtype

dtype('<M8[ns]')

In [346]:
df['ship_date'].isnull().sum()

np.int64(482)

In [347]:
invalid_mask = df['ship_date'] < df['order_date']
invalid_mask.sum()

np.int64(139)

In [348]:
df.loc[invalid_mask,'ship_date'] = df.loc[invalid_mask,'order_date'] + pd.to_timedelta(1,unit='D')

In [349]:
df['order_date'].isnull().sum()

np.int64(1189)

In [350]:
mask = df['order_date'].isnull() & df['ship_date'].notnull()

df.loc[mask,'order_date'] = df.loc[mask,'ship_date'] - pd.to_timedelta(2,unit = 'D')

In [351]:
df['order_date'] = df['order_date'].fillna(df['order_date'].median())

In [352]:
processing_time = (df['ship_date'] - df['order_date']).median()

In [353]:
mask = df['ship_date'].isnull()
df.loc[mask,'ship_date'] = df.loc[mask,'order_date'] + processing_time

In [354]:
df['ship_date'].isnull().sum()

np.int64(0)

In [355]:
df['delivery_date'] = df['delivery_date'].apply(parse_date)

In [356]:
df['delivery_date'].isnull().sum()

np.int64(726)

In [357]:
invalid_mask = df['delivery_date'] < df['ship_date']

df.loc[invalid_mask, 'delivery_date'] = df.loc[invalid_mask, 'ship_date'] + pd.to_timedelta(2, unit='D')


In [358]:
invalid_mask = df['delivery_date'] < df['ship_date']
invalid_mask.sum()

np.int64(0)

In [359]:
delivery_time = (df['delivery_date']- df['ship_date']).median()

In [360]:
mask = df['delivery_date'].isnull()

df.loc[mask,'delivery_date'] = df.loc[mask,'ship_date'] + delivery_time 

In [361]:
df['supplier_location'].value_counts(dropna=False)

supplier_location
Mumbai           710
New York, USA    654
Shanghai, CN     647
NaN              632
DE               627
FRANCE           615
London, UK       615
Name: count, dtype: int64

In [362]:
mapping = {
    'DE': 'Berlin, Germany',
    'FRANCE': 'Paris, France'
}

df['supplier_location'] = df['supplier_location'].replace(mapping)

In [363]:
df['supplier_location'] = df['supplier_location'].str.strip()

In [364]:
df['supplier_location'] = df.groupby('supplier_name')['supplier_location']\
    .transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x))

In [435]:
df[df['supplier_country'].isnull()][['supplier_location']].value_counts()

supplier_location
Mumbai               788
Name: count, dtype: int64

In [437]:
df['supplier_country'].isnull().sum()

np.int64(0)

In [436]:
df.loc[df['supplier_city'] == 'Mumbai', 'supplier_country'] = 'India'

In [365]:
df['supplier_location'] = df['supplier_location'].fillna('Unknown')

In [366]:
df['supplier_location'].isnull().sum()

np.int64(0)

In [367]:
df[['supplier_city', 'supplier_country']] = df['supplier_location'].str.split(',', expand=True)

In [368]:
df['supplier_city'] = df['supplier_city'].str.strip()
df['supplier_country'] = df['supplier_country'].str.strip()

In [440]:
df['supplier_country'].value_counts()

supplier_country
India      788
CN         758
USA        746
Germany    658
UK         634
France     594
Name: count, dtype: int64

In [441]:
df['supplier_city'].value_counts()

supplier_city
Mumbai      788
Shanghai    758
New York    746
Berlin      658
London      634
Paris       594
Name: count, dtype: int64

In [369]:
df['warehouse_location'].value_counts()

warehouse_location
WH-Gamma    1153
WH-Beta     1126
Main_Hub    1123
WH-Alpha    1098
Name: count, dtype: int64

In [370]:
df['warehouse_location'] = df['warehouse_location'].str.strip().str.upper()

In [371]:
df['product_category'].value_counts(dropna=False)

product_category
Electronics         927
Office Supplies     913
Industrial Tools    894
Clothing            886
Furniture           880
Name: count, dtype: int64

In [372]:
df['product_category'] = df['product_category'].str.strip().str.title()

In [373]:
df['quantity'].describe()

count    4500.000000
mean      230.034222
std      1331.373071
min        -5.000000
25%        23.000000
50%        51.000000
75%        76.000000
max      9999.000000
Name: quantity, dtype: float64

In [374]:
df['quantity'].value_counts().head()

quantity
 0       87
 9999    82
-5       74
 73      55
 66      54
Name: count, dtype: int64

In [375]:
(df['quantity'] < 0).sum()

np.int64(74)

In [376]:
df = df[df['quantity'] >= 0]

In [377]:
(df['quantity'] == 0).sum()

np.int64(87)

In [378]:
df.loc[df['quantity'] == 9999, 'quantity'] = np.nan

In [379]:
df['quantity'] = df['quantity'].fillna(df['quantity'].median())

In [380]:
Q1 = df['quantity'].quantile(0.25)
Q3 = df['quantity'].quantile(0.75)
IQR = Q3 - Q1

upper_limit = Q3 + 1.5 * IQR
lower_limit = Q1 - 1.5 * IQR

In [381]:
df[(df['quantity'] > upper_limit) | (df['quantity'] < lower_limit)]

,order_id,order_date,ship_date,delivery_date,supplier_name,supplier_location,warehouse_location,product_category,product_id,quantity,...,shipping_cost,shipping_mode,delivery_status,delay_days,payment_method,customer_type,region,returned,supplier_city,supplier_country


In [382]:
df['unit_price'].describe()

count    4226.000000
mean      256.311886
std       142.841404
min        10.080000
25%       132.562500
50%       255.190000
75%       381.542500
max       499.950000
Name: unit_price, dtype: float64

In [383]:
df['unit_price'].value_counts().head()

unit_price
413.40    3
133.13    3
465.18    3
345.40    3
172.59    3
Name: count, dtype: int64

In [384]:
(df['unit_price'] < 0).sum()

np.int64(0)

In [385]:
df = df[df['unit_price'] >= 0]

In [386]:
df['unit_price'].sort_values(ascending=False).head()

3376    499.95
1581    499.92
313     499.84
3908    499.71
716     499.39
Name: unit_price, dtype: float64

In [387]:
Q1 = df['unit_price'].quantile(0.25)
Q3 = df['unit_price'].quantile(0.75)
IQR = Q3 - Q1

upper_limit = Q3 + 1.5 * IQR
lower_limit = Q1 - 1.5 * IQR

In [388]:
df[(df['unit_price'] > upper_limit) | (df['unit_price'] < lower_limit)]

,order_id,order_date,ship_date,delivery_date,supplier_name,supplier_location,warehouse_location,product_category,product_id,quantity,...,shipping_cost,shipping_mode,delivery_status,delay_days,payment_method,customer_type,region,returned,supplier_city,supplier_country


In [389]:
df['unit_price'].isnull().sum()

np.int64(0)

In [390]:
df['calculated_total'] = df['quantity'] * df['unit_price']
df[['total_cost', 'calculated_total']].head()

,total_cost,calculated_total
0,41258.48,41258.48
1,5437.80,5437.80
2,18782.45,18782.45
3,44179.75,44179.75
4,10992.24,10992.24


In [391]:
df['diff'] = df['total_cost'] - df['calculated_total']

(df['diff'].abs() > 1).sum()

np.int64(455)

In [392]:
mismatch = df[abs(df['diff']) > 1]

mismatch[['quantity', 'unit_price', 'total_cost', 'calculated_total', 'diff']].head(10)

,quantity,unit_price,total_cost,calculated_total,diff
24,50.0,141.26,1412458.74,7063.00,1405395.74
26,10.0,210.80,2326.00,2108.00,218.00
48,40.0,70.35,3049.00,2814.00,235.00
57,58.0,113.06,6946.48,6557.48,389.00
61,98.0,440.44,43604.12,43163.12,441.00
65,31.0,453.04,14223.24,14044.24,179.00
67,96.0,269.65,26195.40,25886.40,309.00
68,100.0,71.24,7529.00,7124.00,405.00
87,39.0,352.09,14184.51,13731.51,453.00
91,50.0,340.74,3407059.26,17037.00,3390022.26


In [393]:
df.loc[abs(df['diff']) > 1, 'total_cost'] = df['calculated_total']

In [394]:
(df['total_cost'] - (df['quantity'] * df['unit_price'])).abs().sum()

np.float64(1.126508664128778e-09)

In [395]:
df.drop(['calculated_total', 'diff'], axis=1, inplace=True)

In [396]:
df['shipping_cost'].describe()

count    3802.000000
mean       52.622002
std        27.582703
min         5.010000
25%        28.742500
50%        52.275000
75%        76.765000
max       100.000000
Name: shipping_cost, dtype: float64

In [397]:
df['shipping_cost'].value_counts().head()

shipping_cost
61.30    4
74.91    4
77.77    4
84.89    4
33.23    4
Name: count, dtype: int64

In [398]:
(df['shipping_cost'] < 0).sum()

np.int64(0)

In [399]:
(df['shipping_cost'] == 0).sum()

np.int64(0)

In [400]:
df['shipping_cost'].isnull().sum()

np.int64(424)

In [401]:
df['shipping_cost'] = df.groupby('shipping_mode')['shipping_cost'].transform(
    lambda x: x.fillna(x.median())
)

In [402]:
df['delay_days'].describe()

count    3382.000000
mean        6.422235
std         5.167214
min        -2.000000
25%         2.000000
50%         6.000000
75%        11.000000
max        15.000000
Name: delay_days, dtype: float64

In [403]:
df['delay_days'].isnull().sum()

np.int64(844)

In [404]:
df['delay_days'].value_counts().head()

delay_days
 1.0     206
 13.0    202
 0.0     199
 2.0     196
-1.0     195
Name: count, dtype: int64

In [405]:
(df['delay_days'] < 0).sum()

np.int64(361)

In [406]:
df['delay_days'] = (df['delivery_date'] - df['ship_date']).dt.days

In [407]:
df['delay_days'].isnull().sum()

np.int64(0)

In [408]:
df[df['delay_days'] > 100][['order_id','ship_date','delivery_date','delay_days']].head()


,order_id,ship_date,delivery_date,delay_days
19,ORD-86507,2022-12-19,2023-04-07,109
34,ORD-82487,2022-12-19,2023-05-08,140
41,ORD-78879,2022-12-19,2023-04-14,116
101,ORD-44741,2022-12-19,2023-08-04,228
270,ORD-58944,2022-12-19,2023-07-20,213


In [409]:
df = df[df['delay_days'] <= 60]

In [410]:
df['returned'].value_counts(dropna=False)

returned
NaN    1415
No     1382
Yes    1381
Name: count, dtype: int64

In [411]:
df['returned'].fillna('No',inplace=True)

/var/folders/f0/c9kqrb7x1gb7px6zxpmwlcy00000gn/T/ipykernel_7897/3589808479.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['returned'].fillna('No',inplace=True)


In [412]:
df['returned'].value_counts()

returned
No     2797
Yes    1381
Name: count, dtype: int64

In [413]:
df['returned_flag'] = df['returned'].map({'Yes': 1, 'No': 0})

In [414]:
df['returned_flag'].mean()

np.float64(0.3305409286740067)

In [415]:
df['payment_method'].value_counts(dropna=False)

payment_method
Net Banking    865
UPI            850
Deb_Card       846
Cash           813
Credit Card    804
Name: count, dtype: int64

In [416]:
df['payment_method'].isnull().sum()

np.int64(0)

In [417]:
df['payment_method'] = df['payment_method'].str.strip().str.lower()

In [418]:
df['payment_method'] = df['payment_method'].replace({
    'credit card': 'Card',
    'deb_card': 'Card',
    'debit card': 'Card',
    
    'upi': 'UPI',
    
    'net banking': 'Net Banking',
    
    'cash': 'COD'
})

In [419]:
df['payment_group'] = df['payment_method'].replace({
    'Card': 'Online',
    'UPI': 'Online',
    'Net Banking': 'Online',
    'COD': 'Offline'
})

In [420]:
df['customer_type'].value_counts(dropna=False)

customer_type
Retail       1428
Corporate    1376
Wholesale    1374
Name: count, dtype: int64

In [421]:
df['customer_segment'] = df['customer_type'].replace({
    'Retail': 'B2C',
    'Corporate': 'B2B',
    'Wholesale': 'B2B'
})

In [422]:
df['is_b2b'] = df['customer_type'].map({
    'Retail': 0,
    'Corporate': 1,
    'Wholesale': 1
})

In [423]:
df['delivery_status'].unique()

array(['Delayed', 'C@nc3ll3d', 'Cancelled', 'On-Time', 'On-Tim3',
       'D3l@y3d'], dtype=object)

In [424]:
df['delivery_status'] = df['delivery_status'].str.lower().str.strip()

In [425]:
df['delivery_status'] = df['delivery_status'].replace({
    'c@nc3ll3d': 'cancelled',
    'd3l@y3d': 'delayed',
    'on-tim3': 'on-time'
})

In [426]:
df['delivery_status'] = df['delivery_status'].str.title()

In [427]:
df['delivery_status'].value_counts()

delivery_status
Cancelled    1440
On-Time      1387
Delayed      1351
Name: count, dtype: int64

In [428]:
df['supplier_city']

0       New York
1       Shanghai
2         Berlin
3       Shanghai
4          Paris
          ...   
4494    Shanghai
4495    New York
4496    Shanghai
4497    New York
4499    New York
Name: supplier_city, Length: 4178, dtype: object

In [429]:
df['supplier_country']

0           USA
1            CN
2       Germany
3            CN
4        France
         ...   
4494         CN
4495        USA
4496         CN
4497        USA
4499        USA
Name: supplier_country, Length: 4178, dtype: object

In [430]:
df.duplicated().sum()


np.int64(0)

In [431]:
df.columns

Index(['order_id', 'order_date', 'ship_date', 'delivery_date', 'supplier_name',
       'supplier_location', 'warehouse_location', 'product_category',
       'product_id', 'quantity', 'unit_price', 'total_cost', 'shipping_cost',
       'shipping_mode', 'delivery_status', 'delay_days', 'payment_method',
       'customer_type', 'region', 'returned', 'supplier_city',
       'supplier_country', 'returned_flag', 'payment_group',
       'customer_segment', 'is_b2b'],
      dtype='object')

In [442]:
df.to_csv("clean_data.csv", index=False)
df.to_csv("clean_data_backup.csv", index=False)